# Using the BaseFunction Module in baseobjects

## Introduction

The `BaseFunction` module provides an abstract class that implements the structure for creating function-like callable objects. It extends `BaseCallable` to create callable objects that behave like functions by directly implementing functionality in methods, while also supporting conversion to methods when bound to instances.

The main purpose of this class is to directly implement the functionality in the methods rather than use wrapped methods. This approach provides better performance and more flexibility when creating custom function-like objects.

When a BaseFunction is accessed through an instance (e.g., `instance.func`), it creates a new method of type `method_type` that is bound to the instance. This allows BaseFunction objects to behave like regular functions when called directly, but like methods when accessed through an instance attribute.

> **Note:** For decorator functionality, use the `BaseDecorator` class from the `baseobjects.functions` package instead. `BaseDecorator` is specifically designed for creating decorators with extended functionality.

This tutorial covers:
- Understanding the purpose and design of `BaseFunction`
- Creating custom function-like objects with direct implementation
- Converting functions to methods when bound to instances
- Binding functions to instances and attributes

**Prerequisites:**
- Basic understanding of Python's functions and methods
- Familiarity with Python's descriptor protocol
- Knowledge of the `BaseCallable` and `BaseMethod` classes from the baseobjects package

### Table of Contents

- [Importing the Module](#Importing-the-Module)
- [Core Functionality](#Core-Functionality)
- [Module Interaction](#Module-Interaction)
- [Advanced Features](#Advanced-Features)
- [Examples](#Examples)
- [API Highlights](#API-Highlights)
- [Troubleshooting / FAQs](#Troubleshooting-/-FAQs)
- [Conclusion and Next Steps](#Conclusion-and-Next-Steps)

## Importing the Module

In [68]:
from baseobjects.bases.basecallable import BaseFunction, BaseMethod

## Core Functionality

The `BaseFunction` class is an abstract class that extends `BaseCallable` to create callable objects that behave like functions by directly implementing functionality in methods. It also supports conversion to methods when bound to instances, providing dual behavior.

### Key Features

1. **Direct Implementation**: Allows you to directly implement functionality in methods
2. **Function-like Behavior**: Behaves like a regular function when called directly
3. **Method Conversion**: Converts to a method when accessed through an instance
4. **Customizable Method Type**: Allows customizing the type of method created when binding
5. **Attribute Binding**: Can bind to instances and set itself as an attribute on the instance

Let's create a simple function-like object using `BaseFunction` with direct implementation:

In [69]:
# Create a custom function by subclassing BaseFunction
class Greeter(BaseFunction):
    """A simple greeting function."""

    my_name = "Greeter"

    def __call__(self, name):
        """Generate a greeting for the given name."""
        return f"Hello, {name}! I'm {self.my_name}!"


# Create an instance of our custom function
greeter = Greeter()

# Call the function directly
print(greeter("World"))

# Check the docstring
print(f"Docstring: {greeter.__doc__}")

Hello, World! I'm Greeter!
Docstring: A simple greeting function.


### Converting to a Method

One of the key features of `BaseFunction` is its ability to convert to a method when accessed through an instance. This allows it to behave like a regular function when called directly, but like a method when accessed through an instance attribute.

In [70]:
# Define a class with a BaseFunction attribute
class Greeter(BaseFunction):
    """A function that generates greetings."""

    def __init__(self, greeting_prefix="Hello") -> None:
        super().__init__()
        self.greeting_prefix = greeting_prefix

    def __call__(self, instance, other):
        """Generate a greeting for another person."""
        return f"{self.greeting_prefix}, {other}! My name is {instance.name}."


# Create a class that will use our function
class Person:
    def __init__(self, name) -> None:
        self.name = name

    # Assign the BaseFunction as a class attribute
    greet = Greeter()


# Create a Person instance
alice = Person("Alice")

# Call the method through the instance
# This will automatically convert the BaseFunction to a method bound to alice
print(alice.greet("Bob"))

Hello, Bob! My name is Alice.


### Customizing Method Type

`BaseFunction` allows customizing the type of method created when binding to an instance through the `method_type` attribute. By default, this is `BaseMethod`, but it can be set to any class that implements the method binding protocol.

In [71]:
# Define a custom method class
class LoggingMethod(BaseMethod):
    """A method that logs its calls."""

    def __call__(self, *args, **kwargs):
        print(f"Calling {self.__wrapped__.__name__} with args: {args}, kwargs: {kwargs}")
        result = super().__call__(*args, **kwargs)
        print(f"Result: {result}")
        return result


# Create a BaseFunction with direct implementation
class Calculator(BaseFunction):
    """A function that performs calculations."""

    def __init__(self) -> None:
        super().__init__()
        # Set the custom method type
        self.method_type = LoggingMethod

    def __call__(self, instance, x, y, operation="add"):
        """Perform a calculation."""
        if operation == "add":
            return x + y
        elif operation == "subtract":
            return x - y
        elif operation == "multiply":
            return x * y
        elif operation == "divide":
            return x / y
        else:
            msg = f"Unknown operation: {operation}"
            raise ValueError(msg)


# Define a class with the BaseFunction attribute
class MathTool:
    def __init__(self, name) -> None:
        self.name = name

    calc = Calculator()


# Create a MathTool instance
math_tool = MathTool("MyCalc")

# Call the method through the instance
# This will create a LoggingMethod bound to math_tool
result = math_tool.calc(5, 3, operation="multiply")

### Binding to Instances

`BaseFunction` provides methods for explicitly binding to instances (`bind`) and for binding to an instance and setting the result as an attribute on the instance (`bind_to_attribute`). These methods give fine-grained control over the binding process.

In [72]:
# Create a BaseFunction with direct implementation
class FullNameGenerator(BaseFunction):
    """A function that generates full names."""

    def __call__(self, instance):
        """Get the full name of a person."""
        return f"{instance.first_name} {instance.last_name}"


# Create a BaseFunction instance
func_full_name = FullNameGenerator()


# Define a class
class Person:
    def __init__(self, first_name, last_name) -> None:
        self.first_name = first_name
        self.last_name = last_name


# Create a Person instance
john = Person("John", "Doe")

# Bind the function to the instance
bound_full_name = func_full_name.bind(john, Person)

# Call the bound method
print(f"Full name: {bound_full_name()}")

# Bind the function to another instance and set it as an attribute
jane = Person("Jane", "Smith")
func_full_name.bind_to_attribute(jane, Person, "full_name")

# Call the method through the instance
print(f"Jane's full name: {jane.full_name()}")

Full name: John Doe
Jane's full name: Jane Smith


## Module Interaction

The `BaseFunction` module interacts with other modules in the baseobjects package, particularly `BaseCallable` and `BaseMethod`. These interactions provide enhanced functionality for function-like objects.

### Interaction with BaseCallable

`BaseFunction` inherits from `BaseCallable`, which means it also inherits all the functionality of `BaseCallable`, including coroutine support and attribute preservation:

In [73]:
# Create a BaseFunction with direct implementation and custom attributes
class MathFunction(BaseFunction):
    """A function that performs mathematical operations."""

    # Define custom attributes
    author = "BaseObjects Team"
    version = "1.0.0"

    def __call__(self, x, y):
        """Add two numbers."""
        return x + y


# Create a BaseFunction instance
func_add = MathFunction()

# Call the function
print(f"5 + 3 = {func_add(5, 3)}")

# Check if the custom attributes are preserved
print(f"Author: {func_add.author}")
print(f"Version: {func_add.version}")

5 + 3 = 8
Author: BaseObjects Team
Version: 1.0.0


### Interaction with BaseMethod

`BaseFunction` creates instances of `BaseMethod` (or the class specified in `method_type`) when binding to instances. This allows for customizing the behavior of bound methods:

In [74]:
# Create a BaseFunction with direct implementation
class NameFormatter(BaseFunction):
    """A function that formats names."""

    def __call__(self, instance, title="Mr./Ms."):
        """Format the name with a title."""
        # Access the bound instance's attributes
        return f"{title} {instance.first_name} {instance.last_name}"


# Create a BaseFunction instance
func_format = NameFormatter()


# Define a class
class Contact:
    def __init__(self, first_name, last_name) -> None:
        self.first_name = first_name
        self.last_name = last_name


# Create a Contact instance
contact = Contact("John", "Doe")

# Bind the function to the instance
bound_format = func_format.bind(contact, Contact)

# Check the type of the bound method
print(f"Type of bound_format: {type(bound_format)}")
print(f"Is instance of BaseMethod: {isinstance(bound_format, BaseMethod)}")

# Call the bound method
print(bound_format())
print(bound_format("Dr."))

Type of bound_format: <class 'baseobjects.bases.basecallable.BaseMethod'>
Is instance of BaseMethod: True
Mr./Ms. John Doe
Dr. John Doe


## Advanced Features

The `BaseFunction` class provides several advanced features that make it powerful for creating custom function-like objects with direct implementation.

### Creating a Custom Function Class

you can create your own custom function class by inheriting from `BaseFunction` and directly implementing functionality in the `__call__` method:

In [75]:
class LoggingFunction(BaseFunction):
    """A function that logs its calls."""

    def __init__(self, log_prefix="FUNCTION", *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.log_prefix = log_prefix
        self.call_count = 0

    def __call__(self, x, y):
        """Multiply two numbers and log the operation."""
        self.call_count += 1

        # Directly implement the functionality
        result = x * y

        print(f"{self.log_prefix} #{self.call_count}: multiply({x}, {y})")
        print(f"{self.log_prefix} #{self.call_count} result: {result}")

        return result


# Create a LoggingFunction
logging_multiply = LoggingFunction(log_prefix="MULTIPLY")

# Call the function
result1 = logging_multiply(5, 3)
result2 = logging_multiply(7, 2)

print(f"Total calls: {logging_multiply.call_count}")

MULTIPLY #1: multiply(5, 3)
MULTIPLY #1 result: 15
MULTIPLY #2: multiply(7, 2)
MULTIPLY #2 result: 14
Total calls: 2


### Creating a Direct Implementation Function Factory

`BaseFunction` can be used to create function factories that directly implement functionality:

In [76]:
class MathFunctionFactory(BaseFunction):
    """A factory that creates math functions with direct implementation."""

    def __init__(self, operation="add") -> None:
        super().__init__()
        self.operation = operation
        self.__name__ = operation
        self.__doc__ = f"Perform {operation} on two numbers."

    def __call__(self, x, y):
        """Perform a math operation on two numbers."""
        # Directly implement the functionality based on the operation
        if self.operation == "add":
            return x + y
        elif self.operation == "subtract":
            return x - y
        elif self.operation == "multiply":
            return x * y
        elif self.operation == "divide":
            return x / y
        else:
            msg = f"Unknown operation: {self.operation}"
            raise ValueError(msg)


# Create some math functions
add_func = MathFunctionFactory("add")
subtract_func = MathFunctionFactory("subtract")
multiply_func = MathFunctionFactory("multiply")
divide_func = MathFunctionFactory("divide")

# Call the functions
print(f"5 + 3 = {add_func(5, 3)}")
print(f"5 - 3 = {subtract_func(5, 3)}")
print(f"5 * 3 = {multiply_func(5, 3)}")
print(f"5 / 3 = {divide_func(5, 3)}")

# Check the attributes
print(f"Name of add_func: {add_func.__name__}")
print(f"Docstring of add_func: {add_func.__doc__}")

5 + 3 = 8
5 - 3 = 2
5 * 3 = 15
5 / 3 = 1.6666666666666667
Name of add_func: add
Docstring of add_func: Perform add on two numbers.


### Creating a Performance Timer

> **Note:** For decorator functionality, use the `BaseDecorator` class from the `baseobjects.functions` package instead. `BaseDecorator` is specifically designed for creating decorators with extended functionality.

We can use `BaseFunction` to create a callable that directly implements performance timing functionality:

In [77]:
import time


class PerformanceTimer(BaseFunction):
    """A callable that measures the execution time of operations."""

    def __init__(self, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.execution_times = []
        self.total_executions = 0

    def __call__(self, n):
        """Calculate the sum of squares from 0 to n-1 and measure performance."""
        # Start timing
        start_time = time.time()

        # Directly implement the calculation
        result = 0
        for i in range(n):
            result += i * i

        # End timing
        end_time = time.time()
        execution_time = end_time - start_time

        # Store timing information
        self.execution_times.append(execution_time)
        self.total_executions += 1

        print(f"Operation took {execution_time:.6f} seconds to execute")
        return result

    def get_average_time(self):
        """Get the average execution time."""
        if not self.execution_times:
            return 0
        return sum(self.execution_times) / len(self.execution_times)

    def get_total_time(self):
        """Get the total execution time."""
        return sum(self.execution_times)

    def get_stats(self):
        """Get performance statistics."""
        return {
            "total_executions": self.total_executions,
            "average_time": self.get_average_time(),
            "total_time": self.get_total_time(),
            "min_time": min(self.execution_times) if self.execution_times else 0,
            "max_time": max(self.execution_times) if self.execution_times else 0,
        }


# Create an instance of our performance timer
timer = PerformanceTimer()

# Call the timer with different inputs
result1 = timer(100000)
print(f"Result 1: {result1}")

result2 = timer(500000)
print(f"Result 2: {result2}")

result3 = timer(1000000)
print(f"Result 3: {result3}")

# Get performance statistics
stats = timer.get_stats()
print("\nPerformance Statistics:")
for key, value in stats.items():
    if "time" in key:
        print(f"  {key}: {value:.6f} seconds")
    else:
        print(f"  {key}: {value}")

Operation took 0.007670 seconds to execute
Result 1: 333328333350000
Operation took 0.041420 seconds to execute
Result 2: 41666541666750000
Operation took 0.065880 seconds to execute
Result 3: 333332833333500000

Performance Statistics:
  total_executions: 3
  average_time: 0.038323 seconds
  total_time: 0.114970 seconds
  min_time: 0.007670 seconds
  max_time: 0.065880 seconds


## Examples

Let's explore some practical examples of using the `BaseFunction` module with direct implementation.

### Creating Property Accessors

We can use `BaseFunction` to directly implement property accessor methods:

In [78]:
class PropertyAccessor(BaseFunction):
    """A function that directly implements property get/set functionality."""

    def __init__(self, property_name, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.property_name = property_name
        self.__name__ = property_name
        self.__doc__ = f"Get or set the {property_name} property."

    def __call__(self, instance, value=None):
        """Get or set the property value."""
        # Get operation (no value provided)
        if value is None:
            return getattr(instance, f"_{self.property_name}", None)

        # Set operation (value provided)
        setattr(instance, f"_{self.property_name}", value)
        return value

    def get_property_info(self, instance=None):
        """Get information about this property."""
        if instance is None:
            return {
                "property_name": self.property_name,
                "bound": False,
            }
        else:
            return {
                "property_name": self.property_name,
                "bound": True,
                "instance_type": type(instance).__name__,
                "current_value": getattr(instance, f"_{self.property_name}", None),
            }


# Define a class with property accessors
class User:
    def __init__(self, name=None, email=None) -> None:
        self._name = name
        self._email = email

        # Create additional property accessors dynamically if needed
        # self.phone = PropertyAccessor("phone", instance=self)

    # Create property accessors directly as class attributes
    name = PropertyAccessor("name")
    email = PropertyAccessor("email")
    address = PropertyAccessor("address")  # This property starts as None


# Create a User instance
user = User("John", "john@example.com")

# Get property values
print(f"Name: {user.name()}")
print(f"Email: {user.email()}")
print(f"Address: {user.address()}")  # Should be None initially

# Set property values
user.name("Jane")
user.email("jane@example.com")
user.address("123 Main St")

# Get the updated values
print(f"Updated name: {user.name()}")
print(f"Updated email: {user.email()}")
print(f"Updated address: {user.address()}")

# Get property information
print("\nProperty information:")
print(f"Name property: {user.name.get_property_info()}")
print(f"Email property: {user.email.get_property_info()}")
print(f"Address property: {user.address.get_property_info()}")

Name: John
Email: john@example.com
Address: None
Updated name: Jane
Updated email: jane@example.com
Updated address: 123 Main St

Property information:
Name property: {'property_name': 'name', 'bound': False}
Email property: {'property_name': 'email', 'bound': False}
Address property: {'property_name': 'address', 'bound': False}


### Creating a Polymorphic Handler

We can use `BaseFunction` to directly implement polymorphic behavior that adapts based on the instance it's bound to:

In [79]:
class PolymorphicHandler(BaseFunction):
    """A function that directly implements polymorphic behavior based on instance type."""

    def __init__(self, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.handler_registry = {}
        self.call_history = []

    def register_handler(self, class_type, handler_func):
        """Register a handler function for a specific class type."""
        self.handler_registry[class_type.__name__.lower()] = handler_func
        return self  # Allow method chaining

    def __call__(self, instance, *args, **kwargs):
        """Handle the call based on the bound instance type."""
        # Access the bound instance through self.__self__ when bound
        instance_type = instance.__class__.__name__.lower()

        # Record this call
        call_record = {
            "instance_type": instance.__class__.__name__,
            "instance": instance,
            "args": args,
            "kwargs": kwargs,
        }

        # Find the appropriate handler
        if instance_type in self.handler_registry:
            handler = self.handler_registry[instance_type]
            call_record["handler_used"] = instance_type
        else:
            handler = self.default_handler
            call_record["handler_used"] = "default"

        # Call the handler
        result = handler(instance, *args, **kwargs)
        call_record["result"] = result

        # Store the call record
        self.call_history.append(call_record)

        return result

    def default_handler(self, instance, *args, **kwargs) -> str:
        """Default handler for unregistered instance types."""
        return f"Default handling for {instance.__class__.__name__}: {getattr(instance, 'name', 'Unknown')}"

    def get_call_stats(self):
        """Get statistics about calls processed."""
        if not self.call_history:
            return {"total_calls": 0}

        stats = {"total_calls": len(self.call_history)}

        # Count calls by instance type
        type_counts = {}
        for record in self.call_history:
            instance_type = record["instance_type"]
            type_counts[instance_type] = type_counts.get(instance_type, 0) + 1

        stats["calls_by_type"] = type_counts
        return stats


# Define some classes
class Person:
    def __init__(self, name) -> None:
        self.name = name


class Company:
    def __init__(self, name, industry) -> None:
        self.name = name
        self.industry = industry


class Organization:
    def __init__(self, name) -> None:
        self.name = name


# Create a polymorphic handler and register handlers for different types
handler = PolymorphicHandler()
handler.register_handler(Person, lambda self, *args, **kwargs: f"Processing Person: {self.name}")
handler.register_handler(Company, lambda self, *args, **kwargs: f"Processing Company: {self.name} ({self.industry})")

# Create instances
person = Person("John")
company = Company("Acme Inc.", "Technology")
org = Organization("Charity")

# Bind the handler to each instance
person.process = handler.bind(person, Person)
company.process = handler.bind(company, Company)
org.process = handler.bind(org, Organization)

# Call the methods
print(person.process())
print(company.process())
print(org.process())  # Will use the default handler

# Get call statistics
stats = handler.get_call_stats()
print("\nCall Statistics:")
print(f"Total calls: {stats['total_calls']}")
print("Calls by type:")
for type_name, count in stats["calls_by_type"].items():
    print(f"  {type_name}: {count}")

Processing Person: John
Processing Company: Acme Inc. (Technology)
Default handling for Organization: Charity

Call Statistics:
Total calls: 3
Calls by type:
  Person: 1
  Company: 1
  Organization: 1


### Creating a Direct Implementation Chain of Responsibility

We can use `BaseFunction` to directly implement a chain of responsibility pattern:

In [80]:
class ChainFunction(BaseFunction):
    """A function that directly implements the chain of responsibility pattern."""

    def __init__(self, handler_name="default", *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.handler_name = handler_name
        self.next_handler = None

    def set_next(self, handler):
        """Set the next handler in the chain."""
        self.next_handler = handler
        return handler

    def __call__(self, x, y):
        """Handle the request based on the input types."""
        try:
            # Directly implement the handler logic based on handler_name
            if self.handler_name == "integer":
                if not (isinstance(x, int) and isinstance(y, int)):
                    msg = "Not integers"
                    raise TypeError(msg)
                return f"Integer division: {x // y}"

            elif self.handler_name == "float":
                if not (isinstance(x, (int, float)) and isinstance(y, (int, float))):
                    msg = "Not numbers"
                    raise TypeError(msg)
                return f"Float division: {x / y}"

            elif self.handler_name == "string":
                if not (isinstance(x, str) and isinstance(y, str)):
                    msg = "Not strings"
                    raise TypeError(msg)
                return f"String concatenation: {x + y}"

            else:  # default handler
                return f"Default handler: {x} and {y}"

        except Exception:
            # If this handler can't handle it, pass to the next handler
            if self.next_handler:
                return self.next_handler(x, y)
            else:
                # If there's no next handler, re-raise the exception
                raise


# Create the chain of responsibility with direct implementation
integer_handler = ChainFunction("integer")
float_handler = ChainFunction("float")
string_handler = ChainFunction("string")
default_handler = ChainFunction("default")

# Set up the chain
integer_handler.set_next(float_handler).set_next(string_handler).set_next(default_handler)

# Use the chain
try:
    print(integer_handler(10, 3))
    print(integer_handler(10.5, 3.2))
    print(integer_handler("Hello, ", "World!"))
    print(integer_handler([1, 2], [3, 4]))
except Exception as e:
    print(f"Unhandled exception: {e}")

Integer division: 3
Float division: 3.28125
String concatenation: Hello, World!
Default handler: [1, 2] and [3, 4]


## API Highlights

Here are the key components of the `BaseFunction` module API:

### BaseFunction
- `__init__(func=None, *args, **kwargs)`: Initialize a new BaseFunction instance
- `method_type`: The type of method to create when binding this function to an instance
- `bind(instance, owner=None)`: Creates a method of this function which is bound to another object
- `bind_to_attribute(instance=None, owner=None, name=None)`: Creates a method of this function which is bound to another object and sets it as an attribute

For more detailed information, consult the full API documentation.

## Troubleshooting / FAQs

### Q: What's the difference between BaseFunction and BaseMethod?

A: `BaseFunction` is designed to behave like a function when called directly, but can be converted to a method when accessed through an instance. `BaseMethod`, on the other hand, is designed to always behave like a method and maintains a reference to the instance it's bound to. `BaseFunction` by default creates a builtin `Method` but can be changed to return a `BaseMethod` instance when it's accessed through an instance.

### Q: How does BaseFunction handle binding to instances?

A: When a `BaseFunction` is accessed through an instance (e.g., `instance.func`), it uses the `__get__` method to bind. By default, it used `bind_builtin` to create bound method for speed. Alternatively, `bind` can be assigned to `__get__` to bind methods of type `method_type` (by default, `BaseMethod`) that is bound to the instance. This allows `BaseFunction` objects to behave like regular functions when called directly, but like methods when accessed through an instance attribute.

### Q: Can I customize the type of method created when binding?

A: Yes, you can customize the type of method created when binding by setting the `method_type` attribute of a `BaseFunction` instance. By default, this is `BaseMethod`, but it can be set to any class that implements the method binding protocol.

### Q: Should I use BaseFunction for creating decorators?

A: No, for decorator functionality, use the `BaseDecorator` class from the `baseobjects.functions` package instead. `BaseDecorator` is specifically designed for creating decorators with extended functionality. BaseFunction is designed for directly implementing functionality in methods rather than wrapping existing functions.

## Conclusion and Next Steps

In this tutorial, we've explored the `BaseFunction` module and its primary class, `BaseFunction`. We've seen how this class extends `BaseCallable` to create callable objects that behave like functions by directly implementing functionality in methods, while also supporting conversion to methods when bound to instances.

The main purpose of `BaseFunction` is to directly implement the functionality in methods rather than use wrapped methods. This approach provides better performance and more flexibility when creating custom function-like objects.

> **Note:** For decorator functionality, use the `BaseDecorator` class from the `baseobjects.functions` package instead. `BaseDecorator` is specifically designed for creating decorators with extended functionality.

### Next Steps

- Explore the examples in the baseobjects package that demonstrate more advanced uses of `BaseFunction`
- Try creating your own custom function classes by extending `BaseFunction` and directly implementing functionality
- Experiment with different method types by customizing the `method_type` attribute
- Check out the `BaseDecorator` module for creating decorators
- Consult the full API documentation for more detailed information on the `BaseFunction` module